<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Supervised_Learning_lektioner/Lab_3_Lektion_5_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💳 Supervised Learning – Lektion 5: Fraud Detection

**Målgrupp:** Gymnasiet, 16 år  
**Tid:** ca 60 minuter  
**Mål:** Applicera ML på ett verkligt problem, förstå klassoimbalans, jämföra modeller, justera threshold

---

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/  
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, 
                               precision_score, recall_score, f1_score,
                               roc_curve, auc)
from sklearn.preprocessing import StandardScaler

print("✅ Alla bibliotek importerade!")

---
## 💳 Del 1 – Verkligt Problem: Kreditkorts-bedrägeri

Föreställ dig att du jobbar på en bank. Din uppgift:

> *"Hitta alla kreditkorts-bedrägerier innan de skadar kunden!"*

### Datasetet vi använder:

```
📊 Kaggle Credit Card Fraud Dataset:
   284,807 transaktioner (kreditkortsköp)
   492 bedrägerier (0.17% av alla!)
   
   Features: V1-V28 (anonymiserade PCA-features)
              Amount (köpsumma)
              Time   (sekunder sedan första transaktion)
   
   Label: Class (0 = Legitimt, 1 = Bedrägeri)
```

### Varför är det här ett svårt problem?

```
🟢 Legitimt:   284,315 transaktioner  (99.83%)
🔴 Bedrägeri:      492 transaktioner  ( 0.17%)
```

**Datasetet är extremt obalanserat!**

---

### 🧩 Quiz 5.1

Om en modell alltid säger "Legitimt" (förutsäger aldrig bedrägeri):

1. Vad är accuracy? → ___ %
2. Vad är recall för bedrägerier? → ___ %
3. Är detta en användbar modell? Varför?

In [ ]:
# ============================================================
# LADDA FRAUD-DATA FRÅN KAGGLE (VIA URL)
# ============================================================

print("⏳ Laddar data... (kan ta 30-60 sekunder)")
print()

try:
    url = 'https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv'
    fraud_df = pd.read_csv(url)
    print(f"✅ Data laddad!")
except Exception as e:
    # Fallback: skapa syntetisk liknande data för demo-syfte
    print("⚠️ Kunde inte ladda data från internet.")
    print("   Skapar syntetisk demo-data istället...")
    np.random.seed(42)
    n_legit = 5000
    n_fraud = 50
    n_total = n_legit + n_fraud
    
    # Skapa 28 PCA-liknande features
    fraud_df = pd.DataFrame(
        np.random.randn(n_total, 28),
        columns=[f'V{i}' for i in range(1, 29)]
    )
    fraud_df['Amount'] = np.abs(np.random.exponential(50, n_total))
    fraud_df['Time'] = np.arange(n_total)
    fraud_df['Class'] = 0
    
    # Bedrägerier har annorlunda mönster
    fraud_idx = np.random.choice(n_total, n_fraud, replace=False)
    fraud_df.loc[fraud_idx, 'Class'] = 1
    fraud_df.loc[fraud_idx, 'V1'] += 3
    fraud_df.loc[fraud_idx, 'V3'] -= 2
    fraud_df.loc[fraud_idx, 'Amount'] *= 3
    
    n_legit = n_total - n_fraud
    print(f"   Syntetisk data: {n_total} transaktioner")

n_total = len(fraud_df)
n_fraud = (fraud_df['Class'] == 1).sum()
n_legit = (fraud_df['Class'] == 0).sum()

print()
print(f"📊 Dataset-statistik:")
print(f"   Totalt:    {n_total:,} transaktioner")
print(f"   Legitima:  {n_legit:,} ({n_legit/n_total:.2%})")
print(f"   Bedrägerier: {n_fraud:,} ({n_fraud/n_total:.2%})")
print()
print(f"   Kolumner: {list(fraud_df.columns[:5])} ... + {fraud_df.shape[1]-5} till")
print(f"   Storlek:  {fraud_df.shape}")

In [ ]:
# ============================================================
# VISUALISERA KLASSOIMBALANSEN
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Stapeldiagram
bars = axes[0].bar(['🟢 Legitimt', '🔴 Bedrägeri'], 
                    [n_legit, n_fraud], 
                    color=['#4CAF50', '#F44336'],
                    edgecolor='black', linewidth=0.7)
axes[0].set_title('Klassfördelning (absolut)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Antal transaktioner')
for bar, val in zip(bars, [n_legit, n_fraud]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# Procentuellt
pct_legit = n_legit / n_total * 100
pct_fraud = n_fraud / n_total * 100
wedges, texts, autotexts = axes[1].pie(
    [n_legit, n_fraud], 
    labels=[f'🟢 Legitimt\n{pct_legit:.1f}%', f'🔴 Bedrägeri\n{pct_fraud:.2f}%'],
    colors=['#4CAF50', '#F44336'],
    autopct='%1.2f%%',
    startangle=90,
    pctdistance=0.75
)
axes[1].set_title('Klassfördelning (procent)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"⚠️ OBS: För varje bedrägeri finns det ungefär {n_legit//n_fraud} legitima transaktioner!")
print()
print("💭 Tänk dig att du måste hitta 1 röd godis i en skål med", n_legit//n_fraud, "gröna!")

---
## 🤔 Del 3 – Problemet med Klassoimbalans

Låt oss se vad som händer om vi bara tränar en enkel modell utan att tänka på balansen.

In [ ]:
# ============================================================
# TRÄNA EN "NAIV" MODELL (UTAN HÄNSYN TILL OBALANS)
# ============================================================

# Förbered data
features = [col for col in fraud_df.columns if col != 'Class']
X_fraud = fraud_df[features].values
y_fraud = fraud_df['Class'].values

# Normalisera Amount och Time
scaler = StandardScaler()
X_fraud_scaled = X_fraud.copy()
X_fraud_scaled[:, -2] = scaler.fit_transform(X_fraud[:, -2].reshape(-1, 1)).ravel()
X_fraud_scaled[:, -1] = scaler.fit_transform(X_fraud[:, -1].reshape(-1, 1)).ravel()

# Dela upp data
X_tr, X_te, y_tr, y_te = train_test_split(
    X_fraud_scaled, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

print("📊 Train/Test Split:")
print(f"   Träning: {len(y_tr):,} transaktioner ({(y_tr==1).sum()} bedrägerier)")
print(f"   Test:    {len(y_te):,} transaktioner ({(y_te==1).sum()} bedrägerier)")
print()
print("🔄 Tränar enkel beslutsträd-modell (max_depth=5)...")

naiv_model = DecisionTreeClassifier(max_depth=5, random_state=42)
naiv_model.fit(X_tr, y_tr)
naiv_pred = naiv_model.predict(X_te)

naiv_acc = accuracy_score(y_te, naiv_pred)
naiv_recall = recall_score(y_te, naiv_pred, zero_division=0)
naiv_prec = precision_score(y_te, naiv_pred, zero_division=0)
naiv_f1 = f1_score(y_te, naiv_pred, zero_division=0)

print(f"✅ Träning klar!")
print()
print(f"📊 Resultat för NAIV modell:")
print(f"   Accuracy:          {naiv_acc:.2%}")
print(f"   Precision (fraud): {naiv_prec:.2%}")
print(f"   Recall (fraud):    {naiv_recall:.2%}  ← Hittar vi bedrägerier?")
print(f"   F1-score (fraud):  {naiv_f1:.2%}")
print()

cm_naiv = confusion_matrix(y_te, naiv_pred)
print(f"   Confusion Matrix:")
print(f"   [[{cm_naiv[0,0]:5d}, {cm_naiv[0,1]:5d}],   (Legitima)")
print(f"    [{cm_naiv[1,0]:5d}, {cm_naiv[1,1]:5d}]]   (Bedrägerier)")
print()
print(f"   ⚠️ Av {(y_te==1).sum()} bedrägerier hittade modellen bara {cm_naiv[1,1]}!")

---
## 🏆 Del 4 – Jämför Modeller

Nu ska vi jämföra FLERA modeller och se vilken som bäst hittar bedrägerier!

Kör cellen och välj modell med dropdown-menyn.

In [ ]:
# ============================================================
# INTERAKTIV MODELL-JÄMFÖRELSE
# ============================================================

output_models = widgets.Output()
trained_models = {}

modell_info = {
    'Beslutsträd (enkelt)': {
        'model': DecisionTreeClassifier(max_depth=5, random_state=42),
        'color': '#FF9AA2', 'desc': 'Enkel, snabb men begränsad'
    },
    'Beslutsträd (djupt)': {
        'model': DecisionTreeClassifier(max_depth=20, random_state=42),
        'color': '#FFB347', 'desc': 'Kan overfitta'
    },
    'Random Forest': {
        'model': RandomForestClassifier(n_estimators=50, max_depth=10, 
                                        random_state=42, n_jobs=-1),
        'color': '#4CAF50', 'desc': 'Kombination av många träd'
    },
    'Logistisk Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'color': '#87CEEB', 'desc': 'Enkel linjär modell'
    },
    'Viktat Beslutsträd': {
        'model': DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
        'color': '#DDA0DD', 'desc': 'Viktar ovanliga klasser högre!'
    }
}

def trana_och_visa(modell_namn):
    with output_models:
        clear_output(wait=True)
        
        if modell_namn not in trained_models:
            print(f"🔄 Tränar: {modell_namn}...")
            m = modell_info[modell_namn]['model']
            m.fit(X_tr, y_tr)
            trained_models[modell_namn] = m
            print(f"✅ Klar!")
        else:
            m = trained_models[modell_namn]
        
        pred = m.predict(X_te)
        
        acc = accuracy_score(y_te, pred)
        prec = precision_score(y_te, pred, zero_division=0)
        rec = recall_score(y_te, pred, zero_division=0)
        f1 = f1_score(y_te, pred, zero_division=0)
        cm = confusion_matrix(y_te, pred)
        
        color = modell_info[modell_namn]['color']
        desc = modell_info[modell_namn]['desc']
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Confusion matrix
        ax = axes[0]
        labels = [['TN\n(Rätt legitimt)', 'FP\n(Falskt alarm)'],
                  ['FN\n(Missat bedrägeri)', 'TP\n(Hittat bedrägeri!)']]
        bg_colors = [['#C8E6C9', '#FFCDD2'], ['#FFCDD2', '#C8E6C9']]
        
        for i in range(2):
            for j in range(2):
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1, 
                                            color=bg_colors[i][j], alpha=0.6))
        
        im = ax.imshow(cm, cmap='YlOrRd', aspect='auto', alpha=0.3)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Förutsatt Legitimt', 'Förutsatt Bedrägeri'], fontsize=10)
        ax.set_yticklabels(['Faktiskt Legitimt', 'Faktiskt Bedrägeri'], fontsize=10)
        ax.set_title(f'Confusion Matrix\n{modell_namn}', fontsize=11, fontweight='bold')
        
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f'{cm[i,j]:,}\n({labels[i][j]})', 
                       ha='center', va='center', fontsize=10, fontweight='bold')
        
        # Metrics
        ax2 = axes[1]
        metrics = {'Accuracy': acc, 'Precision\n(Fraud)': prec,
                   'Recall\n(Fraud)': rec, 'F1\n(Fraud)': f1}
        bars = ax2.bar(metrics.keys(), [v*100 for v in metrics.values()],
                       color=color, edgecolor='black', linewidth=0.5, alpha=0.9)
        ax2.set_ylim(0, 115)
        ax2.set_ylabel('Värde (%)', fontsize=11)
        ax2.set_title(f'Evalueringsmått\n({desc})', fontsize=11, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
        
        for bar, val in zip(bars, metrics.values()):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        n_fraud_test = (y_te == 1).sum()
        print(f"🎯 Resultat för {modell_namn}:")
        print(f"   Accuracy:              {acc:.2%}")
        print(f"   Precision (bedrägeri): {prec:.2%}  ← Precision av flaggade bedrägerier")
        print(f"   Recall (bedrägeri):    {rec:.2%}  ← Andel hittade bedrägerier")
        print(f"   F1-score (bedrägeri):  {f1:.2%}")
        print()
        print(f"   Av {n_fraud_test} bedrägerier hittades: {cm[1,1]} ({cm[1,1]/n_fraud_test:.0%})")
        print(f"   Falskt alarm (FP): {cm[0,1]} ({cm[0,1]/(cm[0,0]+cm[0,1]):.2%} av legitima)")

modell_dropdown = widgets.Dropdown(
    options=list(modell_info.keys()),
    value='Beslutsträd (enkelt)',
    description='Välj modell:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

widgets.interactive(trana_och_visa, modell_namn=modell_dropdown)

---
### ✏️ Övning 4.1 – Jämför modellerna

Testa varje modell och fyll i tabellen:

| Modell | Recall (Fraud) | Precision (Fraud) | F1 |
|--------|---------------|-------------------|-----|
| Beslutsträd (enkelt) | ___ | ___ | ___ |
| Beslutsträd (djupt) | ___ | ___ | ___ |
| Random Forest | ___ | ___ | ___ |
| Logistisk Regression | ___ | ___ | ___ |
| Viktat Beslutsträd | ___ | ___ | ___ |

1. Vilken modell har BÄST recall? → ___
2. Vilken modell har BÄST precision? → ___
3. Varför hjälper `class_weight='balanced'`? → ___

---
## 🎚️ Del 5 – Threshold-justering

Modeller returnerar en **sannolikhet** (0.0 - 1.0) för varje klass.  
Vi väljer sedan en **threshold** för att bestämma om vi klassificerar som bedrägeri eller inte.

```
Sannolikhet för bedrägeri:
0.0 ──────────────────── 1.0
     Legitimt → │ ← Bedrägeri
             threshold
```

- **Låg threshold (0.3):** Fler flaggas som bedrägeri → Högt Recall, Lågt Precision
- **Hög threshold (0.8):** Färre flaggas → Lågt Recall, Högt Precision

Prova slider nedan och se trade-off:en!

In [ ]:
# ============================================================
# INTERAKTIV THRESHOLD-JUSTERING
# ============================================================

# Använd Random Forest (bäst av de testade)
if 'Random Forest' not in trained_models:
    print("🔄 Tränar Random Forest...")
    rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    trained_models['Random Forest'] = rf
    print("✅ Klar!")
else:
    rf = trained_models['Random Forest']

# Hämta sannolikheter
fraud_proba = rf.predict_proba(X_te)[:, 1]  # Sannolikhet för bedrägeri

output_thresh = widgets.Output()

def justera_threshold(threshold):
    with output_thresh:
        clear_output(wait=True)
        
        # Klassificera baserat på threshold
        pred_thresh = (fraud_proba >= threshold).astype(int)
        
        acc = accuracy_score(y_te, pred_thresh)
        prec = precision_score(y_te, pred_thresh, zero_division=0)
        rec = recall_score(y_te, pred_thresh, zero_division=0)
        f1 = f1_score(y_te, pred_thresh, zero_division=0)
        cm = confusion_matrix(y_te, pred_thresh)
        
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        
        # ── Confusion matrix ────────────────────────────────
        ax = axes[0]
        im = ax.imshow(cm, cmap='YlOrRd', aspect='auto')
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Legitimt', 'Bedrägeri'], fontsize=10)
        ax.set_yticklabels(['Legitimt', 'Bedrägeri'], fontsize=10)
        ax.set_title(f'Confusion Matrix\n(threshold={threshold:.2f})', fontsize=11, fontweight='bold')
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i,j]), ha='center', va='center', 
                       fontsize=14, fontweight='bold',
                       color='white' if cm[i,j] > cm.max()*0.5 else 'black')
        
        # ── Precision/Recall bar ─────────────────────────────
        ax2 = axes[1]
        bars = ax2.bar(['Precision', 'Recall', 'F1'], 
                       [prec*100, rec*100, f1*100],
                       color=['#4CAF50', '#2196F3', '#FF9800'], 
                       edgecolor='black', linewidth=0.5)
        ax2.set_ylim(0, 115)
        ax2.set_title(f'Mått (threshold={threshold:.2f})', fontsize=11, fontweight='bold')
        ax2.set_ylabel('%')
        ax2.grid(True, alpha=0.3, axis='y')
        for bar, val in zip(bars, [prec, rec, f1]):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.0%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
        
        # ── Threshold på sannolikhetsfördelning ──────────────
        ax3 = axes[2]
        legit_proba = fraud_proba[y_te == 0]
        fraud_proba_true = fraud_proba[y_te == 1]
        ax3.hist(legit_proba, bins=40, alpha=0.6, color='#4CAF50', label='Legitimt', density=True)
        ax3.hist(fraud_proba_true, bins=15, alpha=0.7, color='#F44336', label='Bedrägeri', density=True)
        ax3.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold={threshold:.2f}')
        ax3.set_xlabel('Sannolikhet för bedrägeri', fontsize=10)
        ax3.set_ylabel('Täthet', fontsize=10)
        ax3.set_title('Sannolikhetsfördelning', fontsize=11, fontweight='bold')
        ax3.legend(fontsize=9)
        
        plt.tight_layout()
        plt.show()
        
        n_fraud_test = (y_te == 1).sum()
        n_flagged = pred_thresh.sum()
        
        print(f"📊 Threshold = {threshold:.2f}:")
        print(f"   Totalt flaggade som bedrägeri: {n_flagged:,}")
        print(f"   Av {n_fraud_test} bedrägerier hittades: {cm[1,1]} ({cm[1,1]/max(1,n_fraud_test):.0%}) ← Recall")
        print(f"   Av {n_flagged} flaggade var faktiskt bedrägeri: {cm[1,1]} ({prec:.0%}) ← Precision")
        print()
        
        if threshold < 0.3:
            print("⚠️ Låg threshold: Många falskt alarm – kunderna kan bli irriterade!")
        elif threshold > 0.8:
            print("⚠️ Hög threshold: Missar många bedrägerier – kunder drabbas av bedrägeri!")
        else:
            print(f"💡 Vad är viktigast för banken: att hitta alla bedrägerier (Recall) eller undvika falskt alarm (Precision)?")

thresh_slider = widgets.FloatSlider(
    value=0.5, min=0.01, max=0.99, step=0.01,
    description='Threshold:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px'),
    readout_format='.2f'
)

widgets.interactive(justera_threshold, threshold=thresh_slider)

---
## 📈 Del 6 – ROC-kurvan

**ROC-kurvan** (Receiver Operating Characteristic) visar hur Precision och Recall byter plats för ALLA möjliga thresholds.

**AUC** (Area Under Curve) mäter modellens totala förmåga:
- AUC = 1.0 → Perfekt modell
- AUC = 0.5 → Modellen gissar slumpmässigt

In [ ]:
# ============================================================
# ROC-KURVA FÖR ALLA MODELLER
# ============================================================

print("🔄 Beräknar ROC-kurvor för alla modeller...")

fig, ax = plt.subplots(figsize=(8, 6))

roc_colors = {'Beslutsträd (enkelt)': '#FF9AA2',
               'Random Forest': '#4CAF50', 
               'Logistisk Regression': '#87CEEB',
               'Viktat Beslutsträd': '#DDA0DD'}

for modell_namn, color in roc_colors.items():
    if modell_namn not in trained_models:
        m_info = modell_info[modell_namn]
        m = m_info['model']
        m.fit(X_tr, y_tr)
        trained_models[modell_namn] = m
    
    m = trained_models[modell_namn]
    
    try:
        proba = m.predict_proba(X_te)[:, 1]
    except:
        continue
    
    fpr, tpr, _ = roc_curve(y_te, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{modell_namn} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Slumpmässig gissning (AUC=0.5)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')

ax.set_xlabel('False Positive Rate (FPR)\n= Falskt alarm / Alla legitima', fontsize=11)
ax.set_ylabel('True Positive Rate (TPR / Recall)\n= Hittade bedrägerier / Alla bedrägerier', fontsize=11)
ax.set_title('ROC-kurva: Jämförelse av modeller', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.show()

print("📊 AUC-poäng (ju högre desto bättre):")
for modell_namn in roc_colors.keys():
    if modell_namn in trained_models:
        m = trained_models[modell_namn]
        try:
            proba = m.predict_proba(X_te)[:, 1]
            fpr, tpr, _ = roc_curve(y_te, proba)
            roc_auc = auc(fpr, tpr)
            print(f"   {modell_namn:30s}: AUC = {roc_auc:.3f}")
        except:
            pass

---
## 🧩 Quiz – Lektion 5

**Fråga 1:** Vad är "klassoimbalans" och varför är det ett problem i fraud detection?

*Ditt svar:* ___

---

**Fråga 2:** En modell har 99.8% accuracy på fraud-datasetet. Är den bra? Förklara.

*Ditt svar:* ___

---

**Fråga 3:** Vad händer med recall och precision när du sänker threshold från 0.7 till 0.2?

*Ditt svar:* ___

---

**Fråga 4:** Om du var bank-chef, vilken threshold skulle du välja? Motivera.

*Ditt svar:* ___

---

**Fråga 5:** Vad är en ROC-kurva och vad mäter AUC?

*Ditt svar:* ___

---

**Fråga 6:** Varför hjälper `class_weight='balanced'` för obalanserade datasets?

*Ditt svar:* ___

---

**Fråga 7:** Vad är ett "False Negative" i fraud detection-kontexten? Vad innebär det för kunden?

*Ditt svar:* ___

---

**Fråga 8:** Vad är ett "False Positive" i fraud detection? Vad innebär det för kunden?

*Ditt svar:* ___

---

**Fråga 9:** En modell flaggar 1000 transaktioner som bedrägeri. Bara 50 är faktiskt bedrägerier. Vad är precision?

*Ditt svar:* ___

---

**Fråga 10:** Varför är Random Forest ofta bättre än ett enda beslutsträd?

*Ditt svar:* ___

---

**Fråga 11:** Vilken threshold ger bäst balans mellan precision och recall i din modell?

*Ditt svar:* ___

---

**Fråga 12:** Vad är skillnaden mellan ett välbalanserat dataset och ett obalanserat?

*Ditt svar:* ___

---

**Fråga 13:** Om threshold=0, vad förutsäger modellen alltid? Vad blir recall och precision?

*Ditt svar:* ___

---

**Fråga 14:** Om threshold=1, vad förutsäger modellen alltid? Vad blir recall och precision?

*Ditt svar:* ___

---

**Fråga 15:** Sammanfatta kursen: Vad är de 3 viktigaste sakerna du lärt dig?

*Ditt svar:*
1. ___
2. ___
3. ___

---

### 🎉 Grattis – Du har klarat Supervised Learning-kursen!

### ✅ Vad du lärt dig i de 5 lektionerna:

| Lektion | Ämne | Nyckelbegrepp |
|---------|------|--------------|
| 1 | Vad är ML? | Supervised, Unsupervised, Features, Labels |
| 2 | Utforska data | Pandas, Histogram, Scatterplot, Klassbalans |
| 3 | Träna modell | Beslutsträd, Train/Test split, Accuracy, Overfitting |
| 4 | Evaluering | Confusion Matrix, Precision, Recall, F1-score |
| 5 | Verkligt problem | Klassoimbalans, Threshold, ROC-kurva, Random Forest |

### 🚀 Vad händer härnäst?

- 🧠 **Deep Learning** – Neurala nätverk
- 📸 **Computer Vision** – Bildanalys med ML
- 💬 **NLP** – Textanalys och chatbots
- 🎮 **Reinforcement Learning** – Spelande AI